# 03 Baseline Decision Tree Model

This notebook trains the first unrestricted baseline Decision Tree using only the URL-text features created by our own extractor. The test set is not loaded or used here.

## What Is a Decision Tree?

A Decision Tree is a supervised machine-learning model that learns a sequence of yes/no-style rules. For this project, the rules split URL feature values until the model predicts either phishing (`0`) or legitimate (`1`).

## Why Can an Unrestricted Tree Overfit?

If we do not limit the tree depth, the model can keep creating more splits until it memorizes patterns that are too specific to the training data. This can produce excellent training accuracy but weaker validation accuracy.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.tree import DecisionTreeClassifier

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.config import RANDOM_STATE
from src.evaluate import evaluate_model
from src.feature_definitions import FEATURE_NAMES
from src.inspect_data import find_target_column
from src.split_data import TRAIN_FILE, VALIDATION_FILE

## Load Training and Validation Data

Training data is used to fit the model. Validation data is used for early evaluation while keeping the final test set untouched.

In [2]:
train_data = pd.read_csv(TRAIN_FILE, index_col=0)
validation_data = pd.read_csv(VALIDATION_FILE, index_col=0)

target_column = find_target_column(train_data)

x_train = train_data[FEATURE_NAMES]
y_train = train_data[target_column]
x_validation = validation_data[FEATURE_NAMES]
y_validation = validation_data[target_column]

train_data.shape, validation_data.shape

((165056, 27), (35369, 27))

## Train the Unrestricted Baseline Tree

This baseline uses `DecisionTreeClassifier(random_state=42)` with no tuning. It is a starting point, not the final model.

In [3]:
baseline_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
baseline_tree.fit(x_train, y_train)

tree_depth = baseline_tree.get_depth()
number_of_leaves = baseline_tree.get_n_leaves()

tree_depth, number_of_leaves

(25, np.int64(388))

The depth and number of leaves describe model complexity. A deeper tree with many leaves is more flexible, but it may also overfit.

## Training and Validation Accuracy

Training accuracy measures performance on data the model learned from. Validation accuracy measures performance on held-out data used for model selection decisions.

In [4]:
training_metrics = evaluate_model(baseline_tree, x_train, y_train)
validation_metrics = evaluate_model(baseline_tree, x_validation, y_validation)

accuracy_summary = pd.DataFrame(
    {
        "split": ["Training", "Validation"],
        "accuracy": [training_metrics["accuracy"], validation_metrics["accuracy"]],
        "precision_phishing": [
            training_metrics["precision_phishing"],
            validation_metrics["precision_phishing"],
        ],
        "recall_phishing": [
            training_metrics["recall_phishing"],
            validation_metrics["recall_phishing"],
        ],
        "f1_phishing": [
            training_metrics["f1_phishing"],
            validation_metrics["f1_phishing"],
        ],
        "roc_auc_phishing": [
            training_metrics["roc_auc_phishing"],
            validation_metrics["roc_auc_phishing"],
        ],
    }
)

accuracy_summary.round(4)

,split,accuracy,precision_phishing,recall_phishing,f1_phishing,roc_auc_phishing
0,Training,0.9962,0.9994,0.9917,0.9955,0.9982
1,Validation,0.9953,0.9983,0.9908,0.9945,0.9958


For precision, recall, F1-score, and ROC-AUC, phishing (`label 0`) is treated as the positive class because missing a phishing URL is the security-sensitive problem.

## Classification Metrics

The table below shows validation metrics from the classification report. The phishing row is especially important for this project.

In [5]:
validation_report = pd.DataFrame(validation_metrics["classification_report"]).T
validation_report.round(4)

,precision,recall,f1-score,support
Phishing,0.9983,0.9908,0.9945,15142.0000
Legitimate,0.9931,0.9988,0.9959,20227.0000
accuracy,0.9953,0.9953,0.9953,0.9953
macro avg,0.9957,0.9948,0.9952,35369.0000
weighted avg,0.9954,0.9953,0.9953,35369.0000


## Overfitting Discussion

A large gap between training and validation accuracy can indicate overfitting. Here, the unrestricted tree has very high training accuracy and very high validation accuracy. The gap should still be monitored because the tree is fairly deep, and later depth analysis may find a simpler tree with similar validation performance.